# 4. Deploy the policy on the robot machine

Install the pinned environment from notebook 1 on the machine connected to the follower and cameras. Copy the **entire** `pretrained_model` directory produced by notebook 3 to that machine, including the policy processors and configuration. This example uses a local model; no Hub upload is required.

**The following terminal command moves a physical robot.** Secure the follower and clear its reach; keep an operator at the power cutoff. Check the initial pose, camera views, wiring, calibration and object positions before each rollout. Start with a short run and be ready to stop with Ctrl+C or cut motor power. A model can produce unsafe actions; never leave it unattended.

## Prepare the deployment host

Use `lerobot-find-port` and `lerobot-find-cameras opencv` on this machine. Reuse the follower's original calibration ID and its calibration file; calibration files are local to the original machine. If they are absent or you use a different physical follower, calibrate it using notebook 2 before proceeding. Camera indices may differ on the new host, but camera names, physical views, resolution, task and rename mapping must match training.

In an activated terminal, edit these examples:

```bash
export FOLLOWER_PORT=/dev/ttyACM0  # replace with the deployment machine's port
export FOLLOWER_ID=workshop_follower
export MODEL_DIR="$HOME/lerobot-workshop-runs/smolvla/checkpoints/last/pretrained_model"
export TASK='Pick up the cube and place it in the bin.'
export CAMERAS='{top: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 30}, wrist: {type: opencv, index_or_path: 1, width: 640, height: 480, fps: 30}}'
export RENAME_MAP='{"observation.images.top": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}'
```

On a separate deployment host set `MODEL_DIR` to the actual copied directory, not a dangling `last` symlink. Verify that `config.json`, weights and processor files exist there. Set camera indices to the physical top and wrist cameras on this host.

In [ ]:
from pathlib import Path
import torch

model = Path.home() / 'lerobot-workshop-runs' / 'smolvla' / 'checkpoints' / 'last' / 'pretrained_model'
print('Default model configuration exists:', (model / 'config.json').is_file())
print('CUDA available:', torch.cuda.is_available())

Update the check above if `MODEL_DIR` differs. The command below uses CUDA; choose `--device=cpu` on a CPU-only host if supported, but verify that inference meets the necessary control rate before operating the robot. See the [deployment guide](https://huggingface.co/docs/lerobot/en/inference) for device and inference options.

## Supervised short rollout

The leader is not needed for autonomous inference. This runs for at most 10 seconds, without recording an evaluation dataset:

```bash
lerobot-rollout \
  --strategy.type=base \
  --robot.type=so101_follower --robot.port="$FOLLOWER_PORT" --robot.id="$FOLLOWER_ID" \
  --robot.cameras="$CAMERAS" --rename_map="$RENAME_MAP" \
  --policy.path="$MODEL_DIR" --device=cuda \
  --task="$TASK" --duration=10
```

Observe success, failure and actual control cadence across multiple supervised trials, resetting the scene safely between runs. Stop immediately for unexpected motion. For recorded evaluation episodes and other rollout strategies, consult the [deployment guide](https://huggingface.co/docs/lerobot/en/inference).